In [ ]:
!pip install tensorflow

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist

# Load the standard MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

def binarize(images, threshold=128):
    """
    Maps pixels below the threshold to -1, and above to +1.
    We also flatten the 28x28 images into a 1D array of 784 pixels.
    """
    # np.where works like an if/else statement across the whole array
    binarized = np.where(images < threshold, -1, 1)

    # Reshape from (number_of_images, 28, 28) to (number_of_images, 784)
    # astype(np.float32) ensures compatibility with TensorFlow
    return binarized.reshape(len(images), -1).astype(np.float32)

x_train_bin = binarize(x_train)   # Shape: (60000, 784)
x_test_bin  = binarize(x_test)    # Shape: (10000, 784)

print(f"Training data shape: {x_train_bin.shape}")

Training data shape: (60000, 784)


In [ ]:
import tensorflow as tf
from tensorflow import keras

def build_model(hidden_neurons=30):
    """
    Builds a simple 1-hidden-layer dense neural network.
    """
    model = keras.Sequential([
        # Input layer: 784 pixels
        keras.layers.Input(shape=(784,)),

        # Hidden layer: Uses hard_sigmoid to approximate the sign function
        keras.layers.Dense(hidden_neurons, activation='hard_sigmoid'),

        # Output layer: 10 neurons for digits 0-9.
        # Softmax gives us probabilities for standard training.
        keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# We will build the 30-neuron version (DiNN-30)
model_30 = build_model(hidden_neurons=30)

print("Training the continuous network...")
# Train for 10 epochs. 10% of the training data is used for validation.
model_30.fit(x_train_bin, y_train, epochs=10, batch_size=128, validation_split=0.1)

Training the continuous network...
Epoch 1/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8121 - loss: 0.8603 - val_accuracy: 0.9138 - val_loss: 0.4117
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8976 - loss: 0.4020 - val_accuracy: 0.9265 - val_loss: 0.2990
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9114 - loss: 0.3234 - val_accuracy: 0.9347 - val_loss: 0.2527
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9204 - loss: 0.2829 - val_accuracy: 0.9397 - val_loss: 0.2304
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9267 - loss: 0.2585 - val_accuracy: 0.9383 - val_loss: 0.2207
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9327 - loss: 0.2380 - val_accuracy: 0.9442 - val_loss: 0.2019
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9362 - loss: 0.2224 - val_accuracy: 0.9458 - val_loss: 0.1924
Epoch 8/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.940

In [ ]:
def process_weight(w, tau=10):
    """
    Corrected: Multiply by tau, round to the nearest integer, then divide back.
    Example: 0.45 * 10 = 4.5 -> round to 5.0 -> divide by 10 = 0.5
    """
    return np.round(w * tau) / tau

def discretize_model(model, tau=10):
    """
    Extracts all weights from the model, discretizes them, and puts them back.
    """
    new_weights = []
    for layer_weights in model.get_weights():
        new_weights.append(process_weight(layer_weights, tau))

    model.set_weights(new_weights)
    return model

# Discretize our trained model
model_30_dinn = discretize_model(model_30, tau=10)

In [ ]:
def sign_activation(x):
    """ The strict mathematical sign function: returns -1, 0, or 1. """
    return tf.sign(x)

def build_dinn_eval(weights, hidden_neurons=30):
    """
    Builds an evaluation-only model with the strict sign function.
    This model cannot be trained; it is only for checking accuracy.
    """
    inp = keras.Input(shape=(784,))

    # Layer 1
    h = keras.layers.Dense(hidden_neurons, use_bias=True)(inp)

    # Apply strict sign activation via a custom Lambda layer
    h = keras.layers.Lambda(sign_activation)(h)

    # Layer 2 (Output)
    out = keras.layers.Dense(10, use_bias=True)(h)

    m = keras.Model(inp, out)
    m.set_weights(weights)
    return m

# Build the strict evaluation model using our discretized weights
dinn_30_eval = build_dinn_eval(model_30_dinn.get_weights(), 30)

print("\nEvaluating discretized model with exact sign(x) activation...")
# To evaluate, we need to compile the evaluation model first
dinn_30_eval.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
loss, accuracy = dinn_30_eval.evaluate(x_test_bin, y_test, verbose=0)
print(f"Cleartext Accuracy after Discretization: {accuracy * 100:.2f}%")


Evaluating discretized model with exact sign(x) activation...
Cleartext Accuracy after Discretization: 89.44%


In [ ]:
import os

def export_for_openfhe(model, path_prefix, tau=10):
    """
    Extracts weights, converts them to pure integers (W * tau),
    calculates norms to size the C++ message space, and saves as CSVs.
    """
    weights_list = model.get_weights()
    # Keras Dense layers return [Weights, biases]
    W1, b1, W2, b2 = weights_list

    # 1. Convert to pure integers by multiplying by tau and rounding
    W1_int = np.round(W1 * tau).astype(int)
    b1_int = np.round(b1 * tau).astype(int)
    W2_int = np.round(W2 * tau).astype(int)
    b2_int = np.round(b2 * tau).astype(int)

    # 2. Calculate the L1 and L2 norms on the INTEGER weights
    # axis=0 means we sum down the columns (each column is the input vector to one neuron)
    l1_norms = np.sum(np.abs(W1_int), axis=0)
    l2_norms = np.linalg.norm(W1_int, axis=0)

    print(f"\n--- OpenFHE Parameters ---")
    print(f"Layer 1 Max L1 Norm (Message Space Bound 'B'): {l1_norms.max():.0f}")
    print(f"Layer 1 Max L2 Norm (Noise Growth Factor): {l2_norms.max():.2f}")
    print(f"Use B >= {l1_norms.max():.0f} in your OpenFHE configuration.")

    # 3. Export as flat CSVs so C++ can parse them easily
    # delimiter=',' creates standard CSV files.
    # fmt='%d' ensures they are written as plain integers.
    os.makedirs("weights", exist_ok=True)

    np.savetxt(f"weights/{path_prefix}_W1.csv", W1_int, delimiter=',', fmt='%d')
    np.savetxt(f"weights/{path_prefix}_b1.csv", b1_int, delimiter=',', fmt='%d')
    np.savetxt(f"weights/{path_prefix}_W2.csv", W2_int, delimiter=',', fmt='%d')
    np.savetxt(f"weights/{path_prefix}_b2.csv", b2_int, delimiter=',', fmt='%d')

    print(f"Integer weights exported successfully to the /weights folder!")

# Run the export
export_for_openfhe(model_30_dinn, "dinn30", tau=10)


--- OpenFHE Parameters ---
Layer 1 Max L1 Norm (Message Space Bound 'B'): 736
Layer 1 Max L2 Norm (Noise Growth Factor): 37.66
Use B >= 736 in your OpenFHE configuration.
Integer weights exported successfully to the /weights folder!
